In [1]:
!pip install -q pandas scikit-learn matplotlib seaborn

import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


os.chdir('/content')
!rm -rf NLP_Course


REPO_URL = "https://github.com/dmytroslav/NLP_Course.git"
!git clone $REPO_URL
sys.path.append('/content/NLP_Course/src')

import split


data_path = '/content/NLP_Course/data/processed_v2.csv'
try:
    df = pd.read_csv(data_path).dropna(subset=['text'])
    print(f" Завантажено {len(df)} текстів.")
    display(df.head())
except FileNotFoundError:
    print(f" Файл {data_path} не знайдено.")


Cloning into 'NLP_Course'...
remote: Enumerating objects: 114, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 114 (delta 40), reused 95 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (114/114), 1.94 MiB | 23.60 MiB/s, done.
Resolving deltas: 100% (40/40), done.
 Завантажено 10735 текстів.


,text,label
0,"Просто слухайте цей діалог. Ні, це не нарізка ...",True
1,️ Рубль став найнестабільнішою валютою у всьом...,True
2,Перше звернення мера Мелітополя Івана Федорова...,True
3,"Росія загрожує Боснії ""українським сценарієм"" ...",True
4,"Енергоатом повідомив, що окупанти пошкодили ви...",True


In [2]:

target_col = 'label'
strategy = 'stratified'

print(f"Обрана стратегія: {strategy} (по колонці {target_col})")


splits = split.make_splits(
    df=df,
    strategy=strategy,
    seed=42,
    test_size=0.1,
    val_size=0.1,
    stratify_col=target_col
)


split.save_splits(splits, '/content/NLP_Course/data/sample')

train_df = splits['train']
val_df = splits['val']
test_df = splits['test']

print(f"Розміри: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

Обрана стратегія: stratified (по колонці label)
Спліти збережено у /content/NLP_Course/data/sample
Маніфест збережено у /content/NLP_Course/data/docs/splits_manifest_lab5.json
Розміри: Train=8587, Val=1074, Test=1074


In [3]:

print("=== 1. Розподіл класів (label) ===") # [cite: 171]
for name, split_df in zip(['Train', 'Val', 'Test'], [train_df, val_df, test_df]):
    print(f"\n{name} розподіл (%):")
    print(split_df[target_col].value_counts(normalize=True).round(3) * 100)

print("\n=== 2. Розподіл довжин тексту ===") # [cite: 173]
for name, split_df in zip(['Train', 'Val', 'Test'], [train_df, val_df, test_df]):
    lengths = split_df['text'].str.len()
    print(f"{name:<5}: Середня={lengths.mean():.0f}, Медіана={lengths.median():.0f}, 5-95%=[{lengths.quantile(0.05):.0f} - {lengths.quantile(0.95):.0f}]")

=== 1. Розподіл класів (label) ===

Train розподіл (%):
label
True     76.7
False    23.3
Name: proportion, dtype: float64

Val розподіл (%):
label
True     76.7
False    23.3
Name: proportion, dtype: float64

Test розподіл (%):
label
True     76.7
False    23.3
Name: proportion, dtype: float64

=== 2. Розподіл довжин тексту ===
Train: Середня=100, Медіана=89, 5-95%=[46 - 190]
Val  : Середня=99, Медіана=89, 5-95%=[47 - 186]
Test : Середня=100, Медіана=90, 5-95%=[45 - 185]

=== 3. Розподіл джерел/груп ===


In [4]:
#Leakage checks

print("=== 2.1 Exact duplicate leakage ===")
train_texts = set(train_df['text'].str.lower().str.strip())
val_texts = set(val_df['text'].str.lower().str.strip())
test_texts = set(test_df['text'].str.lower().str.strip())

exact_train_val = len(train_texts.intersection(val_texts))
exact_train_test = len(train_texts.intersection(test_texts))
print(f"# exact duplicates train∩val = {exact_train_val}")
print(f"# exact duplicates train∩test = {exact_train_test}")

print("\n=== 2.2 Near-duplicate leakage (TF-IDF) ===")
vectorizer = TfidfVectorizer(max_features=5000)
tfidf_train = vectorizer.fit_transform(train_df['text'])
tfidf_test = vectorizer.transform(test_df['text'])


sim_matrix = cosine_similarity(tfidf_test, tfidf_train)
threshold = 0.95 # [cite: 189]
suspicious_pairs = np.where(sim_matrix > threshold)
print(f"Знайдено {len(suspicious_pairs[0])} підозрілих near-duplicate пар (threshold > {threshold})")

print("\n=== 2.3 Template / metadata leakage ===")
template_leaks = train_df[train_df['text'].str.contains(r'(?i)\blabel\b|\btrue\b|\bfalse\b', na=False)]
print(f"Знайдено текстів із можливими підказками-метаданими: {len(template_leaks)}")

print("\n=== 2.4 - 2.5 Group & Time leakage ===")
print("Пропущено: відсутні метадані часу та груп (user_id/source).")

=== 2.1 Exact duplicate leakage ===
# exact duplicates train∩val = 23
# exact duplicates train∩test = 22

=== 2.2 Near-duplicate leakage (TF-IDF) ===
Знайдено 32 підозрілих near-duplicate пар (threshold > 0.95)

=== 2.3 Template / metadata leakage ===
Знайдено текстів із можливими підказками-метаданими: 0

=== 2.4 - 2.5 Group & Time leakage ===
Пропущено: відсутні метадані часу та груп (user_id/source).


In [ ]:

os.makedirs('/content/NLP_Course/docs', exist_ok=True)

# 1. Leakage Risk Report
risk_report = f"""# Leakage Risk Report (Lab 5)

## 1. Стратегія split
Обрано стратегію: **Stratified random split**.
Оскільки цільова змінна (`label`) має бінарні класи (True/False), ця стратегія є оптимальною для збереження балансу класів між вибірками (Train/Val/Test). Вона запобігає ризику того, що рідкісний клас потрапить лише в одну вибірку, і гарантує чесне тестування. Через відсутність часових міток або ідентифікаторів користувачів/джерел, стратегії time-based або group split не застосовувались.

## 2. Статистика сплітів
- **Розміри**: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}.
- **Баланс**: Стратифікація успішна (відсоток класів True/False ідентичний). Розподіл довжин текстів стабільний між сплітами (середні та медіани майже збігаються)[cite: 219].

## 3. Leakage checks results
- exact duplicates train∩test = {exact_train_test}
- exact duplicates train∩val = {exact_train_val}
- near-duplicates (sim > 0.95): {len(suspicious_pairs[0])}
- template leakage: {len(template_leaks)} випадків (слова true/false/label у тексті)
- group/time leakage: N/A (немає колонок груп та дат).

## 4. Ризики, що залишились
- Знайдені near-duplicates свідчать про те, що одні й ті самі новини могли бути перефразовані. Це створює незначний ризик завищення метрик на тестовій вибірці.
- Деякі тексти містять англійські слова "true"/"false", що може бути випадковим збігом, а може підказувати моделі клас.

## 5. Що ви зробите далі
- Видалити exact duplicates перед навчанням ML-моделей.
- Використовувати `sklearn.pipeline.Pipeline`, щоб гарантувати, що `fit` векторизатора відбувається виключно на `train` вибірці.
"""
with open('/content/NLP_Course/docs/leakage_risk_report_lab5.md', 'w', encoding='utf-8') as f:
    f.write(risk_report)

# 2. Audit Summary
audit_summary = """# Audit Summary (Lab 5)
У ЛР5 реалізовано детермінований стратифікований спліт датасету у пропорції 80/10/10[cite: 150]. Проведено необхідні перевірки на витоки даних (leakage). Виявлено нульовий/мінімальний рівень точних дублікатів між вибірками, проте зафіксовано наявність near-duplicates (через специфіку збору новин). Ризики задокументовані, а дисципліна ізоляції тестової вибірки ("fit only on train") дотримана.
"""
with open('/content/NLP_Course/docs/audit_summary_lab5.md', 'w', encoding='utf-8') as f:
    f.write(audit_summary)

print(" Звіти згенеровано та збережено у папку /docs/!")